# Train Talos Shield Model in Colab

This notebook trains the initial blue-team DistilBERT prompt-injection classifier on `deepset/prompt-injections`, exports it as a Hugging Face model directory, and zips it for download.

After download, extract the folder into `subnet/models/shield_model/` on your local machine so the blue miner can load it immediately at startup.

In [ ]:
!pip install -q transformers datasets huggingface_hub pyarrow accelerate torch

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "/content/shield_model"
ZIP_PATH = "/content/shield_model.zip"
EPOCHS = 2
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
TEST_SIZE = 0.1
SEED = 42

In [ ]:
import os
import shutil
import zipfile

import numpy as np
import pyarrow.parquet as pq
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)

In [ ]:
dataset_path = hf_hub_download(
    repo_id="deepset/prompt-injections",
    filename="data/train-00000-of-00001-9564e8b05b4757ab.parquet",
    repo_type="dataset",
)
table = pq.read_table(dataset_path)
texts = table.column("text").to_pylist()
labels = table.column("label").to_pylist()

dataset = Dataset.from_dict({"text": texts, "label": labels})
split = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.rename_column("label", "labels")
eval_dataset = eval_dataset.rename_column("label", "labels")
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean().item()
    return {"accuracy": accuracy}

training_args = TrainingArguments(
    output_dir="/content/training_output",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for root, _, files in os.walk(OUTPUT_DIR):
        for filename in files:
            full_path = os.path.join(root, filename)
            archive.write(full_path, arcname=os.path.relpath(full_path, OUTPUT_DIR))

print(f"Saved model directory to: {OUTPUT_DIR}")
print(f"Saved zip archive to: {ZIP_PATH}")

In [ ]:
from google.colab import files

files.download(ZIP_PATH)

## Local setup after download

1. Extract `shield_model.zip`.
2. Copy the extracted files into `subnet/models/shield_model/`.
3. Make sure your local `.env` contains:

```bash
SHIELD_MODEL_PATH=./models/shield_model
SHIELD_MODEL_POLL_INTERVAL_SEC=5
```

The blue miner will then load that checkpoint at startup and fine-tune it locally when new entries are appended to `dangerous_prompts.json`.